# 00 Diffusion 数学与前向加噪

目标：先不训练神经网络，只理解 DDPM 的核心变量。

你会看到：
- 为什么扩散模型要逐步加噪；
- `beta`, `alpha`, `alpha_bar` 分别是什么；
- 如何从原图 `x0` 一步采样到任意时刻 `xt`；
- 为什么训练目标通常是预测噪声 `epsilon`。

In [ ]:
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

torch.manual_seed(42)
random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
class DiffusionSchedule:
    def __init__(self, timesteps=300, beta_start=1e-4, beta_end=0.02, device=device):
        self.timesteps = timesteps
        self.betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_bars = torch.sqrt(self.alpha_bars)
        self.sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - self.alpha_bars)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.posterior_variance = self.betas * (1.0 - torch.cat([torch.ones(1, device=device), self.alpha_bars[:-1]])) / (1.0 - self.alpha_bars)


def extract(values, t, x_shape):
    batch_size = t.shape[0]
    out = values.gather(0, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))


def q_sample(x0, t, schedule, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    return (
        extract(schedule.sqrt_alpha_bars, t, x0.shape) * x0
        + extract(schedule.sqrt_one_minus_alpha_bars, t, x0.shape) * noise
    )

## 观察噪声调度

`alpha_bar_t` 越小，说明原始信号保留越少，噪声占比越高。

In [ ]:
schedule = DiffusionSchedule(timesteps=300)

plt.figure(figsize=(8, 4))
plt.plot(schedule.betas.cpu(), label="beta")
plt.plot(schedule.alpha_bars.cpu(), label="alpha_bar")
plt.legend()
plt.title("Noise schedule")
plt.show()

## 用一张手写数字演示前向扩散

如果第一次运行缺少 `torchvision`，请先在终端或 notebook 中安装依赖：

```bash
pip install torch torchvision matplotlib tqdm
```

In [ ]:
from torchvision import datasets, transforms
from torchvision.utils import make_grid

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2 - 1),
])
dataset = datasets.MNIST(root="../data", train=True, download=True, transform=transform)
x0, label = dataset[0]
x0 = x0.unsqueeze(0).to(device)

timesteps_to_show = torch.linspace(0, schedule.timesteps - 1, 10).long().to(device)
images = []
for t in timesteps_to_show:
    xt = q_sample(x0, t.view(1), schedule)
    images.append(xt)

grid = make_grid(torch.cat(images).cpu().clamp(-1, 1), nrow=10, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(14, 2))
plt.imshow(grid.permute(1, 2, 0), cmap="gray")
plt.axis("off")
plt.title(f"Forward diffusion on MNIST label={label}")
plt.show()

## 训练目标

前向过程：

```text
xt = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * epsilon
```

训练一个网络 `epsilon_theta(xt, t)` 去预测真实噪声 `epsilon`，损失就是：

```text
MSE(epsilon, epsilon_theta(xt, t))
```

后续 notebook 会把这个训练目标接到 MLP、U-Net 和条件生成上。